In [3]:
import pandas as pd
import numpy as np

In [4]:
XLS_PATH = '1292.0.55.002_anzsic 2006 - codes and titles.xls'

# The 'Classes' sheet contains all four hierarchy levels (Division > Subdivision > Group > Class).
# Each level occupies a different column: Division=col1, Subdivision=col2, Group=col3, Class=col4.
# The title for each entry is one column to the right of its code.
raw = pd.read_excel(XLS_PATH, sheet_name='Classes', header=None)
raw.head(12)

,0,1,2,3,4,5
0,Australian Bureau of Statistics,NaN,NaN,NaN,NaN,NaN
1,cat.no.1292.0.55.002 Australian and New Zealan...,NaN,NaN,NaN,NaN,NaN
2,"Table 4. ANZSIC 2006 Division, Subdivision, Gr...",NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN
4,"ANZSIC 2006 Division, Subdivision, Group and C...",NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN
6,NaN,A,"Agriculture, Forestry and Fishing",NaN,NaN,NaN
7,NaN,NaN,01,Agriculture,NaN,NaN
8,NaN,NaN,NaN,011,Nursery and Floriculture Production,NaN
9,NaN,NaN,NaN,NaN,0111,Nursery Production (Under Cover)


In [5]:
# Drop the 6-row metadata header and reset index
data = raw.iloc[6:].reset_index(drop=True)

records = []
cur_div = cur_sub = cur_grp = None

for _, row in data.iterrows():
    c1, c2, c3, c4, c5 = row[1], row[2], row[3], row[4], row[5]

    if pd.notna(c1):  # Division
        cur_div = str(c1).strip()
        cur_sub = cur_grp = None
        records.append({'level': 'division', 'code': cur_div, 'title': str(c2).strip(),
                        'division_code': cur_div, 'subdivision_code': None, 'group_code': None})
    elif pd.notna(c2):  # Subdivision
        cur_sub = str(c2).strip()
        cur_grp = None
        records.append({'level': 'subdivision', 'code': cur_sub, 'title': str(c3).strip(),
                        'division_code': cur_div, 'subdivision_code': cur_sub, 'group_code': None})
    elif pd.notna(c3):  # Group
        cur_grp = str(c3).strip()
        records.append({'level': 'group', 'code': cur_grp, 'title': str(c4).strip(),
                        'division_code': cur_div, 'subdivision_code': cur_sub, 'group_code': cur_grp})
    elif pd.notna(c4):  # Class
        records.append({'level': 'class', 'code': str(c4).strip(), 'title': str(c5).strip(),
                        'division_code': cur_div, 'subdivision_code': cur_sub, 'group_code': cur_grp})

df = pd.DataFrame(records, columns=['level', 'code', 'title', 'division_code', 'subdivision_code', 'group_code'])
print(df.shape)
print(df['level'].value_counts())
df.head(10)

(829, 6)
level
class          507
group          217
subdivision     86
division        19
Name: count, dtype: int64


,level,code,title,division_code,subdivision_code,group_code
0,division,A,"Agriculture, Forestry and Fishing",A,NaN,NaN
1,subdivision,01,Agriculture,A,01,NaN
2,group,011,Nursery and Floriculture Production,A,01,011
3,class,0111,Nursery Production (Under Cover),A,01,011
4,class,0112,Nursery Production (Outdoors),A,01,011
5,class,0113,Turf Growing,A,01,011
6,class,0114,Floriculture Production (Under Cover),A,01,011
7,class,0115,Floriculture Production (Outdoors),A,01,011
8,group,012,Mushroom and Vegetable Growing,A,01,012
9,class,0121,Mushroom Growing,A,01,012


In [6]:
# Export to parquet and CSV
df.to_parquet('ANZSIC.parquet', index=False)
df.to_csv('ANZSIC.csv', index=False)
print('Saved ANZSIC.parquet and ANZSIC.csv')

Saved ANZSIC.parquet and ANZSIC.csv
